## Why XGBoost Exists

- As datasets grew bigger and bigger algorithm like Gradient Boosting, Random Forest and many others had problem of less accurate predictions and were very slow on big datasets which was a problem so XGBoost is a library which try to solve this problems and this library has Gradient Boosting as it's core.

- XGBoost is made up of Gradient Boosting and many optimizations on top of it to increase it's performance and speed on all types of data no matter the size. 

### Why choose Gradient Boosting?
- Gradient Boosting was chosen due to many reasons such as :-
    - it's flexibility as it can use any loss function if it is differentiable unlike other models where you are limited in some numbers of loss function.
    - It gives quite good result on any dataset as we have seen in previous notebook.
    - It is robust if regularization are applied correctly and it handles missing value quite well. 

## What makes XGBoost different 

### - Flexibility
- Cross Platform 
- Multi Language Support it is not only limited to python it is also available in java, C, R, C++, etc. and model build in python can easily and directly run or be integrated any other language.
- Supports all kind of ML problems be it regression, binary class classification, multi class classification, time series forecasting or ranking XGBoost performs well on all this types and you can also add you custom type because you can set your own loss function in XGBoost.

### - Speed
- Using this aspects XGBoost achieves its speed :-

    - **Parallel Processing**: Now you might as how can we perform parallel processing it we are using GB as its core as you know GB trains model sequentially to correct its prediction then how can we convert this sequential process in parallel one and the answer is we don't. Parallel processing is not in terms of model building but it is used to train individual model is a sequence. So while making decision tree what we do is we have to select a feature with best split or lowest gini impurity value now when we are finding gini impurity for one feature we can also parallelly process another feature as they are not related this will directly increases performance and this is what XGBoost does and uses all available cores to process that many features at one. We can activate this by setting "n_jobs" hyperparameter to "-1".

    - **Optimized Data Structure** : XGBoost stores data in column blocks manner instead if row block which helps it in parallel processing and this yy yis one of the example it uses many other such optimizations in terms of data structure to increase speed of model.

    - **Cache Awareness** : XGBoost stores frequently used values in cache for easy access and so that model can use it without recomputing it thus increasing speed of the model.
    
    - **Out of Core Computing** : Major problem with model training are if dataset is bigger than capacity of RAM then we cannot train model on that dataset. But XGBoost solve this problem by dividing initial dataset into chunks then train model on each chunk sequentially and most important part models start training where previous chunk left let's say "s1" is state of the model after training on first chunk then model will start training from "s1" state when training on second chunk of dataset and this happens for all the chunks of dataset. We activate this by setting "tree_method" hyperparameter to "hist". So we can apply XGBoost to very big dataset without the constrains of RAM.

    - **Distributed Computing** : You can train single model on different computer (Nodes) this way model can be trained fast as it is similar to Out of Core Computing but main difference is that in distributed Computing you have multiple computers so you can crunch multiple cores as a time. But you have to use some external libraries such as Dask, PySpark, etc.

    - **GPU Computing** : As GPU has more cores though they are less powerful but more in numbers. As we are XGBoost already supports Parallel processing and GPUs are best for parallel processing. That is why you can also use GPU tp train model. We can let model use GPU to train by setting "tree_method" hyperparameter to "gpu_hist".

### - Performance
- Using this ML aspects XGBoost achieves its performance :-

    - **Regularized Learning Objective** : This means that XGBoost uses 'loss function + regularization term' to create next model by default this prevents overfitting and unlike Gradient Boosting we can now prevent overfitting without changing learning rate or grooming a tree. This also helps model to become more general and gives improved performance across all datasets. This also decreases bias of the model as it is more general to dataset.

    - **Handling Missing Value** : XGBoost handles missing value internally. First it makes decision tree which does not include missing values then it checks where missing values will fit and gives least gini impurity this way model make sense of where missing value can go but not its actual value.       

    - **Sparsity Awareness Split Finding** : This is sort of algorithm or mechanism which efficiently handles missing data, zero values, and sparse features (like one-hot encoded variables) during tree construction.  Instead of requiring pre-imputation or ignoring missing values, XGBoost learns a default direction (left or right child) for missing data at each split node as discussed above.

        - The algorithm works by:

            - Separating data into non-missing and missing (or zero) values for the current feature. 
            - Finding the optimal split threshold using only non-missing values. 
            - Evaluating both directions for missing values: calculating the gain if they go to the left child and if they go to the right child. 
            - Choosing the direction that maximizes the gain (reduction in loss), which becomes the learned default.  


    - **Efficient Split Finding** : We are making many decision trees in XGBoost but for making this trees we have to decide split up till not every tree model uses ***Exact Greedy Search*** in this we take average on to consecutive feature for all the feature in column then make split in that average lastly we compare all this split as decide which condition will provides best split or least gini impurity. Though this process gives best result as we are essentially checking every values but this is very slow as we have to make splits on all the values of dataset. That is why we use ***Approximate Tree Learning*** in this we make bins instead of using every values individually this type of training is called ***Histogram based Training***. Though this increases speed of the model it is less accurate since their is no relation between dataset and bins. To increase performance as well we make bins using ***Weighted Quantile Sketch*** in this we will study the distribution of the feature and make bins according to the numbers of instances that lies between them be keep number of instances equal in each bin this types of binning will capture the distribution of dataset thus making our tree more accurate and eventually increasing performance of model directly.

    - **Tree Pruning** : Tree Pruning means cut sorting trees which will reduce overfitting GB provides two methods for pruning post-pruning and pre-pruning. But XGBoost provides many methods such as pre-pruning, post-pruning, their is hyperparameter "gama_value" which decides that when branch should be created and branch is only created it their is significant reduction in loss.Which gives flexibility when pruning decision trees and in return increase performance over many dataset.


## XGBoost for Regression

### Initial Model
- Initial model is simply 0.5 means regardless of what are the target values and what features say prediction is 0.5.

### Decision Trees (all models after first)
- What we used to for making next model (tree) is we uses feature of original data set as feature but target becomes residual of previous model and for finding best split we uses gini impurity or entropy to determine best split conditions while making tree. But in XGBoost we will be making tree to predict residual using features just like in Gradient Boosting but instead of using gini impurity or entropy we uses **Similarity Score** which is calculated by $$ Similarity\ Score = \frac{(sum\ of\ residuals\ in\ a\ node)^2}{(Number\ of\ residuals) + \lambda}\ \ \ ,\ \lambda\ is\ regularization\ parameter$$ Now, that we know what Similarity score is let's see how tree is created each tree starts as single node with all the residuals of previous model in it then we calculate similarity score for that node our goal is to maximize similarity score. Now, we grow our tree for that we take average between two adjacent instances of ordered features and use that average to make a split now we got two nodes again we calculate similarity score. Now, that we have similarity score of leaf nodes we can quantify how much better the leafs cluster similar residual than the root we can do this by calculating gain and we calculate gain by $$ Gain = Left\ node_{similarity\ score} + Right\ node_{similarity\ score} - Root\ node_{similarity\ score} $$ This gain determines which split conditions creates tree (stump) with best clustering in each node and we select this condition as out split condition we repeat this until maximum depth is reached or the number of values in leaf node is less then minimum required for splitting. 
- Now lets talk about tree pruning we cut-off branches whose stump's gain value is less than $\gamma$ which is hyperparameter which could be set. if branches has not been cut-off root too will remain it will not matter if its gain is less than $\gamma$.
- Now let's talk about regularization of we increase values of $\lambda$ while calculating similarity score we will make tree more general and prevent them from overfitting.
- Since we have created tree what will be it's output or prediction of error is $$ Output = \frac{(sum\ of\ residuals\ in\ a\ node)}{(Number\ of\ residuals) + \lambda}\ \ \ ,\ \lambda\ is\ regularization\ parameter$$
- For the final prediction we all predictions form all model to base model but also multiply them with learning rate which is same for all the models.
 
- Note : Thought their are many mays of making tree in XGBoost this was the most common way.

## XGBoost for Classification

### Initial model 
- Similar to XGBoost regression initial model is 0.5 means it has 50% probability of that instance is having that class as it's classification.

### Decision Trees (all the models after first)
- We do every thing similar to we were doing in XGBoost regression while making tree but with simple change in **Similarity Score**. Now, we calculate it by $$Similarity\ Score =  \frac{\sum{(residual)^2}}{\sum{[(previous\ probability) * (1 - previous probability)]} + \lambda}\ \ \ ,\ \lambda\ is\ regularization\ parameter$$ Calculation of gain is also same. While creating tree we also check cover. cover determines minimum number to values required for leaf node to exist or to make a split. in XGBoost classification cover is $$ cover = \sum_{0}^{No.\ of\ residuals\ in\ a\ node}{[(previous\ probability) * (1 - previous probability)]} $$ and for XGBoost regression $cover = No.\ of\ residuals\ in\ a\ node$ which means we can have minimum 1 residual in leaf node for regression while using default cover which indicates cover has no effect in XGBoost regression if left default.

- Pruning a tree is also same if gain is less than $\gamma$ we cut-off the brach and if branch has not been cut-offed then root will remain too no matter what its gain is.

- Since we have created tree what will be it's output or prediction of error is $$ Output = \frac{\sum{(residual)}}{\sum{[(previous\ probability) * (1 - previous probability)]} + \lambda}\ \ \ ,\ \lambda\ is\ regularization\ parameter$$ 
- For final prediction unlike regression we cannot add directly. As our first simple model was probability we will convert that in to log(odds) using this formula $$log(odds) = log(\frac{p}{1-p})$$
Now that we have log(odds) values we can directly add predicted error or output form models to the simple models log(odds) but also multiply them by learning rate. Since we want probability of it being which class we need to convert this log(odds) back to probability using this formula $$Probability = \frac{e^{log(odds)}}{1 + e^{log(odds)}}$$

## Baseline Comparison Setup

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("cleaned_data.csv", index_col="id")

In [2]:
from sklearn.model_selection import train_test_split

x = df.drop(columns=["default_payment_next_month"])
y = df["default_payment_next_month"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

## XGBoost Pipeline

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn import set_config

# set_config(transform_output='pandas')

numeric_feature = [
    "limit_bal", "age", "bill_amt1", "bill_amt2", "bill_amt3", "bill_amt4", "bill_amt5", "bill_amt6", "pay_amt1", "pay_amt2", "pay_amt3", "pay_amt4", "pay_amt5", "pay_amt6"
]

preprocessor = ColumnTransformer(transformers=[("num", StandardScaler(), numeric_feature)], remainder="passthrough")

In [4]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42,
        ))
    ]
)

### Why is depth small?
- This is to create models which is weak learner which are less pron to overfitting and learning on noise and since we are sequentially adding many weak learners it is ideal so that overfitting is prevented and model becomes more generalized.

### Why subsampling is enabled?
- subsampling is enabled to introduce randomness to training data and thus preventing overfitting and making model more general to the dataset.

### Why this already looks like “controlled GB”?
- Because under the hood it is extremely optimized gradient boosting model.

## Cross Validation

In [5]:
from sklearn.model_selection import cross_validate

scoring = ["accuracy", "precision", "recall", "roc_auc"]

xgb_cv = cross_validate(xgb_model, x_train, y_train, cv = 5, scoring=scoring)

for metric in scoring:
    print(f"{metric} : ",xgb_cv[f"test_{metric}"].mean())
    
xgb_cv

accuracy :  0.8207074014572437
precision :  0.6719192473365527
recall :  0.3708527021499831
roc_auc :  0.7821231882866899


{'fit_time': array([0.36970687, 0.18080115, 0.17742372, 0.17566395, 0.17803168]),
 'score_time': array([0.02032161, 0.0227983 , 0.01953173, 0.01793766, 0.02112603]),
 'test_accuracy': array([0.83190824, 0.81167883, 0.82081769, 0.81372549, 0.82540676]),
 'test_precision': array([0.71428571, 0.6366782 , 0.67005076, 0.64      , 0.69858156]),
 'test_recall': array([0.4005655 , 0.3468426 , 0.37358491, 0.36192271, 0.37134779]),
 'test_roc_auc': array([0.78799106, 0.77458532, 0.78106651, 0.78303401, 0.78393903])}

## Compare: GB vs XGB vs RF

|Model                |ROC-AUC           |Recall            |Precision         |
|---------------------|------------------|------------------|------------------|
|Random Forest        |0.7686566649620062|0.3542594206248999|0.6615605274817108|
|Random Forest (Tuned)|0.7696328090333561|0.5580693815987934|0.5065023956194388|
|Gradient Boosting    |0.7829213219630841|0.3678363238667687|0.6789085028257853|
|XGBoost              |0.7821231882866899|0.3708527021499831|0.6719192473365527|

### Did XGBoost improve ROC-AUC?
- It is almost same as Gradient Boosting though better than Random Forest

### Did it stabilize recall?
- Yes, recall definitely stabilized though not enough to directly use it in our business use case.

### Is the gain worth the added complexity?
- Yes, because even with out tuning recall is high so it is definitely worth adding complexity and as it is XGBoost it is significantly faster than Gradient Boosting which is a plus point.

## Early Stopping

In [6]:
from sklearn.metrics import roc_auc_score

xgb_model.set_params(classifier__early_stopping_rounds=20)

preprocessor = xgb_model.named_steps['preprocessor']

x_train_trans = preprocessor.fit_transform(x_train)
x_test_trans = preprocessor.transform(x_test)

xgb_clf = xgb_model.named_steps['classifier']

xgb_clf.fit(
    x_train_trans, y_train,
    verbose=False,
    eval_set=[(x_test_trans, y_test)]
)

train_probs = xgb_clf.predict_proba(x_train_trans)[:,1]
test_probs = xgb_clf.predict_proba(x_test_trans)[:, 1]

roc_auc_score(y_train, train_probs), roc_auc_score(y_test, test_probs)

(0.8073433557670897, 0.7730067600213429)

### What early stopping prevents?
- Early stopping prevents overfitting it automatically halts the process of training when evaluation metrics are not improving for `early_stopping_rounds` iteration.

### Why this is safer than choosing n_estimators blindly?
- Because we are checking for performance increase and if it does not increases for set number of iterations then and than only we halts training process. which is better and safer than blindly choosing n_estimators.

## When XGBoost is worth using

### When XGBoost is superior to sklearn GB
- When dataset is huge and their are missing values in it. also it does not overfits easily.

### When you would not use XGBoost
- When dataset is unstructured and small. also you want simple model.

### What kind of mistakes XGBoost still cannot fix
- XGBoost fails when their is prediction involved which goes beyond training distribution.

## Diagnosing Model Weakness

### Is recall still low?
- Yes, Recall is 3.708 which is  much lower compare to tuned Random Forest model.

### Is precision too low?
- No, Precision is pretty high compare to tuned random forest model.

### Is ROC-AUC plateauing?
- Yes, compare to tuned random forest model ROC-AUC increased.

### Is the model overfitting (train >> CV)?
- No, because we calculated train roc_auc score in "Early Stopping" section which is 0.80 and roc_auc we got form CV is 0.78 which is not that smaller compare to train roc_auc which concludes that model is not overfitting. generalization is good and early stopping is working as it is intended. 

## Parameter Map

| Problem       | Parameters to Adjust                          | Why             |
| ------------- | --------------------------------------------- | --------------- |
| High bias     | max_depth, n_estimators                       | Trees too weak  |
| High variance | min_child_weight, subsample, colsample_bytree | Overfitting     |
| Low recall    | scale_pos_weight, threshold                   | Class imbalance |
| Slow learning | learning_rate + n_estimators                  | Step size       |


## Tuning Model

### Tuning for **Class Imbalance**

In [7]:
neg, pos = y_train.value_counts()
scale_pos_weight = neg/pos
scale_pos_weight

3.519607843137255

In [8]:
xgb_balanced = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=400,
            max_depth=3,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42,
            scale_pos_weight=scale_pos_weight
        ))
    ]
)

In [9]:
scoring = ["roc_auc", "recall", "precision"]

scores = cross_validate(xgb_balanced, x_train, y_train, cv=5, scoring=scoring)

for metric in scoring:
    print(f"{metric} :",scores[f"test_{metric}"].mean())

roc_auc : 0.7799713352561917
recall : 0.6368751444881119
precision : 0.4642388034793473


#### Did recall improve?
- Yes, it went from 0.3708 to 0.6391 which is very huge improvement over default.

#### What happened to precision?
- As expected it went down form 0.6719 to 0.4656 which is major loss in precision. since they are sort of inversely proportional

#### Why this trade-off makes sense
- We are forcing model to learn how to accurately predict "defaults" of True Positives accurately by making it sensitive to positive-class error (defaults) will decrease precision as threshold got lower more instances will be classified as "defaults" even though they are not.

### Tuning for **Variance**

In [10]:
from sklearn.model_selection import cross_val_score

scores = cross_validate(xgb_balanced, x_train, y_train, cv=5, scoring='roc_auc')

scores["test_score"].mean(), scores['test_score'].std()

(np.float64(0.7799713352561917), np.float64(0.004083749183296761))

- Since models does not gives varies that much in performance as its standard deviation of roc_auc is low we do not need to control variance as it is good and model is pretty stable and it generalizes well it does not means it is optimal. 
- The standard deviation of ROC-AUC across folds was used as the primary indicator of variance. 

### Early Stopping

In [11]:
xgb_balanced.set_params(classifier__early_stopping_rounds=30)

preprocessor = xgb_balanced.named_steps["preprocessor"]

x_train_trans = preprocessor.fit_transform(x_train)
x_test_trans = preprocessor.transform(x_test)

xgb_balanced_clf = xgb_balanced.named_steps["classifier"]

xgb_balanced_clf.fit(
    x_train_trans, y_train,
    eval_set=[(x_test_trans, y_test)],
    verbose=True
)

xgb_balanced_clf.best_iteration

[0]	validation_0-logloss:0.67373
[1]	validation_0-logloss:0.65788
[2]	validation_0-logloss:0.64386
[3]	validation_0-logloss:0.63221
[4]	validation_0-logloss:0.62423
[5]	validation_0-logloss:0.61518
[6]	validation_0-logloss:0.60937
[7]	validation_0-logloss:0.60274
[8]	validation_0-logloss:0.59730
[9]	validation_0-logloss:0.59246
[10]	validation_0-logloss:0.58839
[11]	validation_0-logloss:0.58442
[12]	validation_0-logloss:0.58064
[13]	validation_0-logloss:0.57761
[14]	validation_0-logloss:0.57493
[15]	validation_0-logloss:0.57267
[16]	validation_0-logloss:0.57077
[17]	validation_0-logloss:0.56922
[18]	validation_0-logloss:0.56770
[19]	validation_0-logloss:0.56695
[20]	validation_0-logloss:0.56638
[21]	validation_0-logloss:0.56565
[22]	validation_0-logloss:0.56474
[23]	validation_0-logloss:0.56395
[24]	validation_0-logloss:0.56275
[25]	validation_0-logloss:0.56228
[26]	validation_0-logloss:0.56183
[27]	validation_0-logloss:0.56182
[28]	validation_0-logloss:0.56087
[29]	validation_0-loglos

327

### How many trees were actually used?
- 327 trees were actually used we can check this using `best_iteration` we can also confirm this by subtracting early_stopping_rounds (30) from total trees created (357).

### Why early stopping is superior to guessing n_estimators?
- Because it prevent overfitting and stops models form making more trees which is much better than blindly setting `n_estimators` this might leads to model being overfitted.

## XGBoost Tuning — Reviewer Conclusion

In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix

y_pred = xgb_balanced.predict(x_test)
y_prob = xgb_balanced.predict_proba(x_test)[: ,1]

print("Accuracy : ", accuracy_score(y_test, y_pred))
print("Precision : ", precision_score(y_test, y_pred))
print("Recall : ", recall_score(y_test, y_pred))
print("ROC-AUC : ", roc_auc_score(y_test,y_prob))
print("Confusion Matrix : \n", confusion_matrix(y_test, y_pred))

Accuracy :  0.7567161688636743
Precision :  0.4634146341463415
Recall :  0.6304675716440422
ROC-AUC :  0.7751943865677338
Confusion Matrix : 
 [[3699  968]
 [ 490  836]]


### Which parameter had the biggest impact?
- Our goal was to increase recall of the model and `scale_pos_weight` had biggest impact on model in increasing recall which is crucial for our business use case.

### Which parameters barely mattered?
- Increasing `n_estimators` because after setting `early_stopping_rounds` it does not matter how much high your `n_estimator` is because `early_stopping_rounds` will stop model from making more trees after performance has been stabilized for set number of times.

### Is the model now better than tuned RF?
- Definitely because in our use case we need high recall and precision is decent as well and which says that model is better than tuned RF.